In [ ]:
# step5_manifest_check.py

import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from time import sleep
from base64 import b64decode

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === Input/Output paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_keyword_check_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_manifest_check_output.csv"

# === Read input ===
df = pd.read_csv(input_path)

# === Prepare new columns ===
manifest_flags = []
activity_flags = []
standard_flags = []

standard_manifest_paths = [
    "app/src/main/AndroidManifest.xml",
    "src/main/AndroidManifest.xml"
]

for i, row in df.iterrows():
    if row.get("keyword_check") != "pass":
        manifest_flags.append("N/A")
        activity_flags.append("N/A")
        standard_flags.append("N/A")
        continue

    repo = row["full_name"]
    has_manifest = False
    has_activity = False
    standard_manifest_found = False

    # === Step 1: Search repo contents for AndroidManifest.xml ===
    contents_url = f"https://api.github.com/repos/{repo}/git/trees/HEAD?recursive=1"
    r = requests.get(contents_url, headers=get_headers())
    if r.status_code != 200:
        print(f"⚠️ Failed to list files for {repo}")
        manifest_flags.append("no")
        activity_flags.append("no")
        standard_flags.append("no")
        continue

    manifest_paths = []
    files = r.json().get("tree", [])
    for file in files:
        path = file.get("path", "").lower()
        if path.endswith("androidmanifest.xml"):
            manifest_paths.append(path)
            has_manifest = True
            if path in standard_manifest_paths:
                standard_manifest_found = True

    # === Step 2: Check each manifest for <activity> tag ===
    for path in manifest_paths:
        url = f"https://api.github.com/repos/{repo}/contents/{path}"
        r = requests.get(url, headers=get_headers())
        if r.status_code != 200:
            continue
        try:
            content = b64decode(r.json()["content"]).decode("utf-8", errors="ignore")
            if "<activity" in content.lower():
                has_activity = True
                break
        except Exception as e:
            continue

    manifest_flags.append("yes" if has_manifest else "no")
    activity_flags.append("yes" if has_activity else "no")
    standard_flags.append("yes" if standard_manifest_found else "no")

    if i % 50 == 0:
        print(f"🔍 Checked {i+1} repos...")

# === Append to DataFrame ===
df["has_manifest"] = manifest_flags
df["has_activity"] = activity_flags
df["standard_manifest"] = standard_flags

# === Save to output file ===
df.to_csv(output_path, index=False)
print(f"✅ Step 5 complete. Saved to: {output_path}")
